# NB03: Data Analysis

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup
Import packages

In [4]:
import pandas as pd
import plotly.express as px
import numpy as np
import statsmodels.formula.api as smf

Declare constants

In [5]:
# Month of ChatGPT launch
T_0 = pd.Timestamp("2022-11-01")
T_0

Timestamp('2022-11-01 00:00:00')

Load long form dataframe and extract key values for analysis:
- Most recent month with data for all industries

In [6]:
df = pd.read_csv("../data/processed/employment.csv", parse_dates=["date"])
industry_codes = pd.read_csv("../data/reference/industry_code_map.csv")

# Group by industry and most recent date, then take the minimum date
t_end = df.groupby("industry_name")["date"].max().min()

# Number of unique industries in sample
unique_industries = df.naics.nunique()

df.head()

,date,industry_name,industry_code,naics,ai_exposure,employment,sector
0,2026-06-01,Offices of physicians,65621100,6211,1.009907,3053.7,Private education and health services
1,2026-05-01,Offices of physicians,65621100,6211,1.009907,3054.6,Private education and health services
2,2026-04-01,Offices of physicians,65621100,6211,1.009907,3050.7,Private education and health services
3,2026-03-01,Offices of physicians,65621100,6211,1.009907,3049.8,Private education and health services
4,2026-02-01,Offices of physicians,65621100,6211,1.009907,3011.8,Private education and health services


## Data coverage
As I was unable to match a significant number of AIIE scores to BLS series, I check the coverage of my dataset against the expanded out list of AIIE scores.

In [7]:

# Extract the number of covered industries and total employment for each sector
covered = (df[df["date"] == t_end]
    .groupby("sector")
    .agg(industries=("industry_name", "size"),
    employment=("employment", "sum"))
)

# Calculate total number of series in each sector


# Get unique 3-digit aggregate series IDs
agg_codes = df.loc[df["industry_code"].astype(str).str[5] == "0", "industry_code"].unique().tolist()
agg_codes

prefixes = [str(x)[:-3] for x in agg_codes]

prefixes

industry_codes = industry_codes.assign(
    digit_3 = industry_codes["industry_code"].astype(str).str[:-3],
    )
industry_codes.loc[industry_codes["3_dig"].isin(prefixes)]



KeyError: '3_dig'

In [8]:
ss_df =  pd.json_normalize(
    ss_json["Results"]["series"], 
    record_path="data",
    meta="seriesID")

ss_df = (ss_df
    .assign(
        year=ss_df.year.astype(int),
        month=ss_df.period.str[1:].astype(int),
        industry=ss_df.seriesID.str[3:-2])
    .query(f"year == {t_end.year} & month == {t_end.month}")
)

ss_df = ss_df[["industry", "value"]]
# aiie = pd.read_csv("../data/reference/aiie.csv")
# aiie["naics_2"] = aiie["naics"].astype(str).str[:2]

NameError: name 'ss_json' is not defined

## Employment growth vs AI exposure
Create plot_df for each chart I want to use for analysis. Merge required tables from NB02

In [9]:
first = (
    df.query(f"date == @T_0") # @ used to bring variable into query string
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_start"})
    )

last = (
    df.query(f"date == @t_end") 
    .groupby("industry_name")
    .first()[["employment", "ai_exposure", "sector"]]
    .rename(columns={"employment": "emp_end"})
    )

plot_df = (
    pd.merge(
        left=first, 
        right=last, 
        on=["industry_name", "sector", "ai_exposure"], 
        how="outer")
        .assign(
            emp_growth_pct=lambda x:
            (x.emp_end - x.emp_start) / x.emp_start *100
            )
        .reset_index()
)

fig = px.scatter(
    plot_df,
    x="ai_exposure",
    y="emp_growth_pct",
    color="sector",
    size="emp_start",
    trendline="ols",
    trendline_scope="overall", # Stop plotly from adding a trendline per
    title="Employment Growth vs AI Exposure"
)

fig.show()


Create exposure quartiles:

In [10]:
q1 = df["ai_exposure"].quantile(0.2)
q4 = df["ai_exposure"].quantile(0.8)


df["exposure_group"] = np.where(
    df["ai_exposure"] >= q4, "High exposure", 
    np.where(
        df["ai_exposure"] <= q1, "Low exposure", 
        "Middle"
    )
)

Aggregate employment by month:

In [11]:
plot_df = (df[df["exposure_group"] != "Middle"]
    # .query("date >= @T_0")
    .query("date < @t_end")
    .groupby(["date", "exposure_group"])["employment"]
    .sum()
    .reset_index()
)

plot_df

,date,exposure_group,employment
0,2019-01-01,High exposure,30019.8
1,2019-01-01,Low exposure,27867.5
2,2019-02-01,High exposure,30076.3
3,2019-02-01,Low exposure,27770.7
4,2019-03-01,High exposure,30168.6
...,...,...,...
171,2026-02-01,Low exposure,28627.2
172,2026-03-01,High exposure,34274.7
173,2026-03-01,Low exposure,28692.8
174,2026-04-01,High exposure,34323.7


Convert to an index:

In [12]:
plot_df["employment_index"] = (
    plot_df.groupby("exposure_group")["employment"]
    .transform(lambda x: 100 * x / x.loc[plot_df.loc[x.index, "date"] == T_0].iloc[0])
    )

In [13]:
px.line(
    plot_df,
    x="date",
    y="employment_index",
    color="exposure_group",
    title="Employment Growth in High- and Low-AI Exposure Industries"
)

## Event study: Did AI exposure cause differences in industries' change in employment over time

In [14]:

es_df = df.assign(
    # Take log of employment so that the coefficients reflect percent change in employment
    ln_emp=np.log(df["employment"]),
    # Calculate discrete months after ChatGPT launch
    event_month=(
        (df["date"].dt.year - T_0.year) * 12 
        + (df["date"].dt.month - T_0.month)
        ) 
)

# Make event month categorical so each month receives own regression coefficient
es_df["event_month"] = pd.Categorical(
    es_df["event_month"],
    categories=sorted(es_df["event_month"].unique()),
    ordered=True
)



The following regression is estimated to compare employment in high- and low-AI-exposure industries within the same month, after removing permanent industry differences and economy-wide shocks, and track how that relationship evolves after ChatGPT's launch in November 2022.

ln(emp_it) = α_i + γ_t + Σ β_k(AIExposure_i × 1[event_month = k]) + ε_it
where:
i = industry
t = month
α_i = industry fixed effects
γ_t = month fixed effects
β_k = event-study coefficients

Industry fixed effects controls for structural differences in employment levels between industries.
Month fixed effects control for whole-economy impacts on employment over time (e.g. the Covid recovery, interest rate hiking, change in US policy post-election)

In [18]:

formula='''
ln_emp
~ C(industry_name) 
+ C(date) 
+ ai_exposure:C(event_month, Treatment(reference=-1))
'''
model = smf.ols(
    formula=formula,
    data=es_df
).fit(
    cov_type="cluster",
    cov_kwds={"groups": es_df["industry_name"]}
    )

coef_df = (
    pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values, 
        "se": model.bse.values
    })
)

# Select interaction terms from coefficient df
coef_df = coef_df[coef_df["term"].str.startswith( "ai_exposure:C(event_month" )].copy()

# Extract event month numbers out of terms 
coef_df["event_month"] = coef_df["term"].str.extract(r"\[(-?\d+)\]", expand=False).astype(str)
coef_df

,term,coef,se,event_month
280,"ai_exposure:C(event_month, Treatment(reference...",0.510606,0.006602,-46
281,"ai_exposure:C(event_month, Treatment(reference...",0.512471,0.006596,-45
282,"ai_exposure:C(event_month, Treatment(reference...",0.512360,0.006489,-44
283,"ai_exposure:C(event_month, Treatment(reference...",0.512401,0.006334,-43
284,"ai_exposure:C(event_month, Treatment(reference...",0.513506,0.006190,-42
...,...,...,...,...
365,"ai_exposure:C(event_month, Treatment(reference...",0.532340,0.006180,39
366,"ai_exposure:C(event_month, Treatment(reference...",0.532313,0.006247,40
367,"ai_exposure:C(event_month, Treatment(reference...",0.532805,0.006264,41
368,"ai_exposure:C(event_month, Treatment(reference...",0.533551,0.006369,42


In [16]:
coef_df["lower"] = coef_df["coef"] - 1.96 * coef_df["se"] 
coef_df["upper"] = coef_df["coef"] + 1.96 * coef_df["se"] 

coef_df = coef_df.sort_values("event_month")

In [17]:
coef_df = coef_df.sort_values("event_month") 

fig = px.line(
    coef_df, 
    x="event_month", 
    y="coef", 
    markers=True, 
    title="Event Study: Employment Effects of AI Exposure", 
    labels={ "event_month": "Months Relative to ChatGPT", "coef": "Coefficient" } ) 
    
# Upper confidence interval   
fig.add_scatter(
    x=coef_df["event_month"], 
    y=coef_df["upper"], 
    mode="lines", 
    line=dict(width=0), 
    showlegend=False, 
    hoverinfo="skip" ) 

# Lower confidence interval + fill 
fig.add_scatter(
    x=coef_df["event_month"],
    y=coef_df["lower"],
    mode="lines",
    fill="tonexty",
    fillcolor="rgba(0,100,255,0.2)",
    line=dict(width=0),
    name="95% CI",
    hoverinfo="skip"
    ) 

# Reference lines 
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
    ) 

fig.add_vline(
    x=0,
    line_dash="dash",
    line_color="red",
    annotation_text="ChatGPT"
    ) 

fig.update_layout(
    template="simple_white",
    width=900,
    height=500
    )
    
fig.show()

In [79]:
[ p for p in model.params.index if "ai_exposure" in p ][:20]

['ai_exposure:C(event_month, Treatment(reference=-1))[-46]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-45]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-44]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-43]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-42]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-41]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-40]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-39]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-38]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-37]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-36]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-35]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-34]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-33]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-32]',
 'ai_exposure:C(event_month, Treatment(reference=-1))[-31]',
 'ai_exposure:C(event_mo

In [80]:
cross_section = (
    df.groupby("industry_name")
    .agg( ln_emp=("ln_emp", "mean"), ai_exposure=("ai_exposure", "first")
    )
)

smf.ols("ln_emp ~ ai_exposure", data=cross_section.reset_index()).fit().params

Intercept      5.619360
ai_exposure    0.241303
dtype: float64

## Sensitivity analysis
With and without 44 and 45 codes: 